# Notebook 00 — Diagnostic de faisabilité (les 6 modèles)

**Projet :** Orange Money — Système ML Fraude & Revenue Assurance  
**Auteur :** Koceila SALEM — RA&FM  
**Objectif :** Répondre factuellement aux questions de faisabilité AVANT tout développement.

## Pipeline de ce notebook
```
CSV brut (13 Go)
   ↓ 1. inspection sur échantillon (structure, types)
   ↓ 2. sélection des colonnes utiles aux 6 modèles
   ↓ 3. lecture par CHUNKS sur le fichier COMPLET (toutes les dates)
   ↓ 4. diagnostic : remplissage, distributions, plage temporelle
   ↓ 5. réponses Q1-Q3 + verdict par modèle
   ↓ 6. conversion Parquet propre (dataset ML)
```

## Questions auxquelles ce notebook répond
| Q | Question | Section |
|---|----------|--------|
| Q1 | Combien de jours couvre la base ? | §4 |
| Q2 | Taux de remplissage des COMMISSIONS_* ? (M4) | §5 |
| Q3 | Présence + remplissage des RECONCILIATION_* ? (M6) | §6 |
| — | Remplissage MSISDN (M2) | §7 |
| — | Distribution TRANSFER_STATUS (M5) | §8 |
| — | Faisabilité M3 (multi-jours ?) | §4 |

## 0. Configuration

In [18]:
import pandas as pd
import numpy as np
import os, time, warnings
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

# ============================================================
# PARAMÈTRES
# ============================================================
CSV_PATH    = r'C:\Users\RQKB6834\Downloads\OM_Koceila\OM_Koceila.csv.csv'
ENCODING    = 'latin-1'
SEPARATEUR  = '|'
CHUNK_SIZE  = 500_000      # lignes par chunk (ajuster selon RAM)
OUTPUT_DIR  = Path('outputs/diagnostic')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'CSV        : {CSV_PATH}')
print(f'Existe     : {os.path.exists(CSV_PATH)}')
if os.path.exists(CSV_PATH):
    print(f'Taille     : {os.path.getsize(CSV_PATH)/(1024**3):.2f} Go')

CSV        : C:\Users\RQKB6834\Downloads\OM_Koceila\OM_Koceila.csv.csv
Existe     : True
Taille     : 13.21 Go


## 1. Inspection sur échantillon — structure & colonnes

On lit seulement 10k lignes pour découvrir la structure sans charger 13 Go.

In [19]:
df_peek = pd.read_csv(
    CSV_PATH, sep=SEPARATEUR, encoding=ENCODING,
    nrows=1_000_000, low_memory=False
)

print(f'Colonnes détectées : {df_peek.shape[1]}')
print(f'Lignes échantillon : {df_peek.shape[0]:,}')
print('\nListe complète des colonnes avec leur type :')
for i, col in enumerate(df_peek.columns, 1):
    exemple = df_peek[col].dropna().iloc[0] if df_peek[col].notna().any() else 'VIDE'
    exemple = str(exemple)[:30]
    print(f'  {i:3d}. {col:<28} | {str(df_peek[col].dtype):<10} | ex: {exemple}')

Colonnes détectées : 80
Lignes échantillon : 1,000,000

Liste complète des colonnes avec leur type :
    1. RECEIVER_USER_ID             | object     | ex: MR1006030850001
    2. SENDER_USER_ID               | object     | ex: PT221022.1054.476808
    3. TRANSACTION_AMOUNT           | float64    | ex: 1000.0
    4. COMMISSIONS_PAID             | float64    | ex: 0.0
    5. COMMISSIONS_RECEIVED         | float64    | ex: 0.0
    6. COMMISSIONS_OTHERS           | float64    | ex: 0.0
    7. SERVICE_CHARGE_RECEIVED      | float64    | ex: 0.0
    8. SERVICE_CHARGE_PAID          | float64    | ex: 0.0
    9. TAXES                        | float64    | ex: 0.0
   10. SERVICE_TYPE                 | object     | ex: RC
   11. TRANSFER_STATUS              | object     | ex: TS
   12. SENDER_PRE_BAL               | float64    | ex: 11199.0
   13. SENDER_POST_BAL              | float64    | ex: 10199.0
   14. RECEIVER_PRE_BAL             | float64    | ex: 277921229.0
   15. RECEIVER_POST_BAL   

In [20]:
profil_colonnes = []

for i, col in enumerate(df_peek.columns, 1):
    exemple = df_peek[col].dropna().iloc[0] if df_peek[col].notna().any() else "VIDE"
    
    profil_colonnes.append({
        "index": i,
        "colonne": col,
        "dtype": str(df_peek[col].dtype),
        "nb_null": df_peek[col].isna().sum(),
        "pct_null": round(df_peek[col].isna().mean() * 100, 2),
        "nb_unique": df_peek[col].nunique(dropna=True),
        "exemple": str(exemple)[:80]
    })

df_profil_colonnes = pd.DataFrame(profil_colonnes)

df_profil_colonnes

,index,colonne,dtype,nb_null,pct_null,nb_unique,exemple
0,1,RECEIVER_USER_ID,object,3439,0.34,177012,MR1006030850001
1,2,SENDER_USER_ID,object,19,0.00,284560,PT221022.1054.476808
2,3,TRANSACTION_AMOUNT,float64,0,0.00,15948,1000.0
3,4,COMMISSIONS_PAID,float64,0,0.00,601,0.0
4,5,COMMISSIONS_RECEIVED,float64,0,0.00,331,0.0
5,6,COMMISSIONS_OTHERS,float64,0,0.00,650,0.0
6,7,SERVICE_CHARGE_RECEIVED,float64,0,0.00,965,0.0
7,8,SERVICE_CHARGE_PAID,float64,0,0.00,11,0.0
8,9,TAXES,float64,0,0.00,1,0.0
9,10,SERVICE_TYPE,object,0,0.00,12,RC


In [21]:
# =============================================================
# MAPPING EXACT DES COLONNES — basé sur le diagnostic réel
# (80 colonnes, échantillon 1M lignes)
# Chaque colonne est annotée avec son % de remplissage réel
# =============================================================

cols = df_peek.columns.tolist()

# Format : 'nom_colonne' : pct_rempli (100 - pct_null observé)
# ✅ = exploitable | ⚠️ = partiel | ❌ = inutilisable (quasi vide)

COLONNES_PAR_MODELE = {

    # ── M1 — Détection de fraude (non supervisé) ──────────────
    # Toutes les colonnes clés sont remplies à ~100%
    'M1_fraude': {
        'TRANSACTION_AMOUNT'  : '✅ 100%',
        'SENDER_PRE_BAL'      : '✅ 100%',
        'SENDER_POST_BAL'     : '✅ 100%',
        'RECEIVER_PRE_BAL'    : '✅ 100%',
        'RECEIVER_POST_BAL'   : '✅ 100%',
        'TRANSFER_STATUS'     : '✅ 100%',
        'SERVICE_TYPE'        : '✅ 100%',
        'TRANSFER_SUBTYPE'    : '✅ 100%',
        'CREATED_ON'          : '✅ 100%',
        'SENDER_USER_ID'      : '✅ 100%',
        'SENDER_CITY'         : '✅ 99%',
        'GATEWAY_TYPE'        : '✅ 99.6%',
        'TRANSACTION_TAG'     : '✅ 100%',
    },

    # ── M2 — Réseaux de mules (graphe) ────────────────────────
    # ⚠️ INITIATOR_MSISDN/OTHER_MSISDN quasi vides → utiliser USER_ID
    'M2_mules': {
        'SENDER_USER_ID'      : '✅ 100%   (NŒUD émetteur)',
        'RECEIVER_USER_ID'    : '✅ 99.7%  (NŒUD récepteur)',
        'TRANSACTION_AMOUNT'  : '✅ 100%   (poids arc)',
        'SENDER_CITY'         : '✅ 99%',
        'RECEIVER_CITY'       : '⚠️ 69%',
        'TRANSACTION_TAG'     : '✅ 100%',
        'OTHER_MSISDN'        : '❌ 27%    (ne pas utiliser comme clé)',
        'INITIATOR_MSISDN'    : '❌ 0.07%  (INUTILISABLE)',
    },

    # ── M3 — SIM Swap / prise de contrôle ─────────────────────
    # ❌ 1 SEUL JOUR + INITIATOR_MSISDN vide → NON FAISABLE en l'état
    'M3_simswap': {
        'ATTEMPT_STATUS'      : '⚠️ 11.6%  (3 valeurs seulement)',
        'ERROR_CODE'          : '⚠️ 5.85%  (94% vides = succès)',
        'SENDER_ACC_STATUS'   : '❌ 1 seule valeur (Y) — inutile',
        'MODIFIED_ON'         : '✅ 100%',
        'SENDER_USER_ID'      : '✅ 100%',
        'INITIATOR_MSISDN'    : '❌ 0.07%  (INUTILISABLE)',
    },

    # ── M4 — Anomalies commissions/frais ──────────────────────
    # ✅ Colonnes présentes à 100% MAIS valeurs souvent à 0
    # → vérifier le % de valeurs NON NULLES (≠ remplissage)
    'M4_commissions': {
        'COMMISSIONS_PAID'        : '✅ 100% rempli (601 val. uniques)',
        'COMMISSIONS_RECEIVED'    : '✅ 100% rempli (331 val. uniques)',
        'COMMISSIONS_OTHERS'      : '✅ 100% rempli (650 val. uniques)',
        'SERVICE_CHARGE_RECEIVED' : '✅ 100% rempli (965 val. uniques)',
        'SERVICE_CHARGE_PAID'     : '⚠️ 100% rempli (11 val. — peu varié)',
        'TAXES'                   : '❌ 1 seule valeur (toujours 0) — INUTILE',
        'SERVICE_TYPE'            : '✅ 100%',
        'CREATED_ON'              : '✅ 100%',
    },

    # ── M5 — Prédiction transactions à risque d'échec ─────────
    # ✅ Cible = TRANSFER_STATUS (label DANS les données, pas besoin CRM)
    'M5_echec': {
        'TRANSFER_STATUS'     : '✅ 100%   (CIBLE : TF=échec)',
        'ERROR_CODE'          : '⚠️ 5.85%  (présent surtout si échec)',
        'SENDER_PRE_BAL'      : '✅ 100%',
        'SERVICE_TYPE'        : '✅ 100%',
        'TRANSFER_SUBTYPE'    : '✅ 100%',
        'ATTEMPT_STATUS'      : '⚠️ 11.6%',
        'GATEWAY_TYPE'        : '✅ 99.6%',
        'SENDER_USER_ID'      : '✅ 100%',
    },

    # ── M6 — Réconciliation automatique ───────────────────────
    # ⚠️ RECONCILIATION_BY/FOR remplis seulement à 0.3%
    # mais ces 0.3% SONT précisément les cas à réconcilier
    'M6_reconcil': {
        'RECONCILIATION_BY'   : '⚠️ 0.30%  (2964 cas — c\'est la cible !)',
        'RECONCILIATION_FOR'  : '⚠️ 0.30%  (3022 cas)',
        'ORIGINAL_REF_NUMBER' : '⚠️ 0.46%  (4576 cas)',
        'EXT_TXN_NUMBER'      : '✅ 28.5%  (≠ EXTERNAL_TRANSACTION_ID)',
        'REFERENCE_NUMBER'    : '⚠️ 28.9%',
        'TRANSFER_ID'         : '✅ 100%   (clé unique)',
        'ACTION_TYPE'         : '✅ 100%   (CREATION/ROLLBACK/...)',
        'TRANSACTION_TAG'     : '✅ 100%',
    },
}

# Affichage structuré
noms_modeles = {
    'M1_fraude'      : 'M1 — Détection fraude',
    'M2_mules'       : 'M2 — Réseaux de mules',
    'M3_simswap'     : 'M3 — SIM Swap',
    'M4_commissions' : 'M4 — Anomalies commissions',
    'M5_echec'       : 'M5 — Prédiction échec',
    'M6_reconcil'    : 'M6 — Réconciliation',
}

print('=' * 70)
print('MAPPING EXACT DES COLONNES PAR MODÈLE (diagnostic réel)')
print('=' * 70)

for modele_key, colonnes in COLONNES_PAR_MODELE.items():
    print(f'\n{noms_modeles[modele_key]}')
    print('-' * 70)
    for col, statut in colonnes.items():
        present = '✓' if col in cols else '✗ ABSENTE'
        print(f'  [{present:9}] {col:<26} {statut}')

# Liste plate des colonnes réellement présentes et exploitables
colonnes_a_analyser = set()
for colonnes in COLONNES_PAR_MODELE.values():
    for col in colonnes:
        if col in cols:
            colonnes_a_analyser.add(col)
colonnes_a_analyser = sorted(colonnes_a_analyser)
print(f'\n\nTotal colonnes uniques exploitables : {len(colonnes_a_analyser)}')

MAPPING EXACT DES COLONNES PAR MODÈLE (diagnostic réel)

M1 — Détection fraude
----------------------------------------------------------------------
  [✓        ] TRANSACTION_AMOUNT         ✅ 100%
  [✓        ] SENDER_PRE_BAL             ✅ 100%
  [✓        ] SENDER_POST_BAL            ✅ 100%
  [✓        ] RECEIVER_PRE_BAL           ✅ 100%
  [✓        ] RECEIVER_POST_BAL          ✅ 100%
  [✓        ] TRANSFER_STATUS            ✅ 100%
  [✓        ] SERVICE_TYPE               ✅ 100%
  [✓        ] TRANSFER_SUBTYPE           ✅ 100%
  [✓        ] CREATED_ON                 ✅ 100%
  [✓        ] SENDER_USER_ID             ✅ 100%
  [✓        ] SENDER_CITY                ✅ 99%
  [✓        ] GATEWAY_TYPE               ✅ 99.6%
  [✓        ] TRANSACTION_TAG            ✅ 100%

M2 — Réseaux de mules
----------------------------------------------------------------------
  [✓        ] SENDER_USER_ID             ✅ 100%   (NŒUD émetteur)
  [✓        ] RECEIVER_USER_ID           ✅ 99.7%  (NŒUD récepteur)

## 2. Sélection des colonnes utiles

On définit explicitement les colonnes à analyser pour chaque modèle.  
**⚠️ Ajuste cette liste selon ce qui est réellement apparu en §1.**

In [22]:
# Colonnes critiques par modèle (à valider avec la sortie §1)
COLS_PAR_MODELE = {
    'M1_fraude'      : ['TRANSACTION_AMOUNT', 'SENDER_PRE_BAL', 'SENDER_POST_BAL',
                        'TRANSFER_STATUS', 'CREATED_ON', 'SENDER_USER_ID', 'SERVICE_TYPE'],
    'M2_mules'       : ['INITIATOR_MSISDN', 'OTHER_MSISDN', 'SENDER_USER_ID',
                        'RECEIVER_USER_ID', 'SENDER_CITY', 'TRANSACTION_TAG'],
    'M3_simswap'     : ['ATTEMPT_STATUS', 'ERROR_CODE', 'INITIATOR_MSISDN',
                        'SENDER_ACC_STATUS', 'CREATED_ON', 'SENDER_USER_ID'],
    'M4_commissions' : ['COMMISSIONS_PAID', 'COMMISSIONS_RECEIVED',
                        'SERVICE_CHARGE_PAID', 'SERVICE_CHARGE_RECEIVED',
                        'TAXES', 'SERVICE_TYPE', 'CREATED_ON'],
    'M5_echec'       : ['ERROR_CODE', 'TRANSFER_STATUS', 'SENDER_PRE_BAL',
                        'SERVICE_TYPE', 'ATTEMPT_STATUS', 'SENDER_ACC_STATUS',
                        'RECEIVER_ACC_STATUS'],
    'M6_reconcil'    : ['RECONCILIATION_BY', 'RECONCILIATION_FOR',
                        'EXTERNAL_TRANSACTION_ID', 'TRANSFER_ID', 'TRANSACTION_TAG'],
}

# Vérifier présence réelle
print('=== VÉRIFICATION PRÉSENCE DES COLONNES ===\n')
colonnes_a_analyser = set()
for modele, liste in COLS_PAR_MODELE.items():
    presentes  = [c for c in liste if c in df_peek.columns]
    manquantes = [c for c in liste if c not in df_peek.columns]
    colonnes_a_analyser.update(presentes)
    pct = len(presentes) / len(liste) * 100
    print(f'{modele:<18} : {len(presentes)}/{len(liste)} présentes ({pct:.0f}%)')
    if manquantes:
        print(f'    ❌ Manquantes : {manquantes}')

colonnes_a_analyser = sorted(colonnes_a_analyser)
print(f'\nTotal colonnes uniques à analyser : {len(colonnes_a_analyser)}')

=== VÉRIFICATION PRÉSENCE DES COLONNES ===

M1_fraude          : 7/7 présentes (100%)
M2_mules           : 6/6 présentes (100%)
M3_simswap         : 6/6 présentes (100%)
M4_commissions     : 7/7 présentes (100%)
M5_echec           : 7/7 présentes (100%)
M6_reconcil        : 4/5 présentes (80%)
    ❌ Manquantes : ['EXTERNAL_TRANSACTION_ID']

Total colonnes uniques à analyser : 24


## 3. Lecture par CHUNKS sur le fichier COMPLET

On parcourt **toutes** les lignes du fichier de 13 Go par blocs de 500k, en accumulant uniquement les statistiques nécessaires (jamais tout en RAM).

Statistiques accumulées :
- Nombre total de lignes
- Remplissage (non-null) par colonne
- Dates min/max + ensemble des jours distincts
- Distribution TRANSFER_STATUS
- Distribution SERVICE_TYPE
- Stats commissions (count non nul, somme)

In [ ]:
# Accumulateurs
stats = {
    'n_total'        : 0,
    'non_null'       : defaultdict(int),
    'jours_distincts': set(),
    'date_min'       : None,
    'date_max'       : None,
    'statut_counts'  : defaultdict(int),
    'service_counts' : defaultdict(int),
    'commission_non_null' : defaultdict(int),
    'commission_somme'    : defaultdict(float),
    'n_chunks'       : 0,
}

# Colonnes à surveiller spécifiquement
COL_DATE    = 'CREATED_ON'
COL_STATUT  = 'TRANSFER_STATUS'
COL_SERVICE = 'SERVICE_TYPE'
COLS_COMM   = ['COMMISSIONS_PAID', 'COMMISSIONS_RECEIVED',
               'SERVICE_CHARGE_PAID', 'SERVICE_CHARGE_RECEIVED', 'TAXES']
COLS_COMM   = [c for c in COLS_COMM if c in df_peek.columns]

print('Démarrage de la lecture par chunks...')
print('(cela peut prendre plusieurs minutes sur 13 Go)\n')
debut = time.time()

reader = pd.read_csv(
    CSV_PATH, sep=SEPARATEUR, encoding=ENCODING,
    chunksize=CHUNK_SIZE, low_memory=False,
    usecols=lambda c: c in colonnes_a_analyser  # ne lit QUE les colonnes utiles
)

for chunk in reader:
    stats['n_chunks'] += 1
    stats['n_total']  += len(chunk)

    # Remplissage par colonne
    for col in chunk.columns:
        stats['non_null'][col] += chunk[col].notna().sum()

    # Dates
    if COL_DATE in chunk.columns:
        dates = pd.to_datetime(chunk[COL_DATE], errors='coerce', dayfirst=True)
        dates_valid = dates.dropna()
        if len(dates_valid) > 0:
            cmin, cmax = dates_valid.min(), dates_valid.max()
            stats['date_min'] = cmin if stats['date_min'] is None else min(stats['date_min'], cmin)
            stats['date_max'] = cmax if stats['date_max'] is None else max(stats['date_max'], cmax)
            stats['jours_distincts'].update(dates_valid.dt.date.unique())

    # Statuts
    if COL_STATUT in chunk.columns:
        for val, cnt in chunk[COL_STATUT].value_counts(dropna=False).items():
            stats['statut_counts'][str(val)] += cnt

    # Services
    if COL_SERVICE in chunk.columns:
        for val, cnt in chunk[COL_SERVICE].value_counts(dropna=False).items():
            stats['service_counts'][str(val)] += cnt

    # Commissions
    for col in COLS_COMM:
        if col in chunk.columns:
            num = pd.to_numeric(chunk[col], errors='coerce')
            stats['commission_non_null'][col] += (num.notna() & (num != 0)).sum()
            stats['commission_somme'][col]    += num.fillna(0).sum()

    # Progress
    if stats['n_chunks'] % 5 == 0:
        print(f"  Chunk {stats['n_chunks']:3d} | {stats['n_total']:,} lignes | {time.time()-debut:.0f}s")

duree = time.time() - debut
print(f"\n✅ TERMINÉ : {stats['n_total']:,} lignes en {duree:.0f}s ({stats['n_chunks']} chunks)")

Démarrage de la lecture par chunks...
(cela peut prendre plusieurs minutes sur 13 Go)

  Chunk   5 | 2,500,000 lignes | 38s
  Chunk  10 | 5,000,000 lignes | 76s


## 4. Q1 — Plage temporelle (faisabilité M3 + M5)

In [ ]:
nb_jours = len(stats['jours_distincts'])

print('=' * 60)
print('Q1 — PLAGE TEMPORELLE')
print('=' * 60)
print(f"Date min        : {stats['date_min']}")
print(f"Date max        : {stats['date_max']}")
print(f"Jours distincts : {nb_jours}")
if nb_jours > 0:
    print(f"\nListe des jours : {sorted(stats['jours_distincts'])[:40]}")

print('\n--- VERDICT FAISABILITÉ ---')
if nb_jours <= 1:
    print('❌ M3 (SIM Swap LSTM)  : NON FAISABLE — 1 seul jour, pas de série temporelle')
    print('⚠️  M5 (échec)          : Faisable mais SANS historique glissant 7j')
    print('⚠️  M2 (mules)          : Graphe sur 1 jour uniquement (snapshot)')
elif nb_jours < 7:
    print(f'❌ M3 (SIM Swap LSTM)  : NON FAISABLE — {nb_jours} jours insuffisants (besoin ~30j)')
    print(f'⚠️  M5 (échec)          : Historique glissant limité à {nb_jours}j')
elif nb_jours < 30:
    print(f'⚠️  M3 (SIM Swap LSTM)  : RISQUÉ — {nb_jours} jours, LSTM peu fiable')
    print(f'✅ M5 (échec)          : Historique 7j OK')
else:
    print(f'✅ M3 (SIM Swap LSTM)  : Faisable — {nb_jours} jours suffisants')
    print(f'✅ M5 (échec)          : Historique 7j OK')

Q1 — PLAGE TEMPORELLE
Date min        : 2025-08-29 10:35:20
Date max        : 2025-09-30 23:59:59
Jours distincts : 32

Liste des jours : [datetime.date(2025, 8, 29), datetime.date(2025, 8, 31), datetime.date(2025, 9, 1), datetime.date(2025, 9, 2), datetime.date(2025, 9, 3), datetime.date(2025, 9, 4), datetime.date(2025, 9, 5), datetime.date(2025, 9, 6), datetime.date(2025, 9, 7), datetime.date(2025, 9, 8), datetime.date(2025, 9, 9), datetime.date(2025, 9, 10), datetime.date(2025, 9, 11), datetime.date(2025, 9, 12), datetime.date(2025, 9, 13), datetime.date(2025, 9, 14), datetime.date(2025, 9, 15), datetime.date(2025, 9, 16), datetime.date(2025, 9, 17), datetime.date(2025, 9, 18), datetime.date(2025, 9, 19), datetime.date(2025, 9, 20), datetime.date(2025, 9, 21), datetime.date(2025, 9, 22), datetime.date(2025, 9, 23), datetime.date(2025, 9, 24), datetime.date(2025, 9, 25), datetime.date(2025, 9, 26), datetime.date(2025, 9, 27), datetime.date(2025, 9, 28), datetime.date(2025, 9, 29), da

## 5. Q2 — Remplissage des COMMISSIONS (faisabilité M4)

In [ ]:
print('=' * 60)
print('Q2 — COLONNES COMMISSIONS (M4)')
print('=' * 60)

if COLS_COMM:
    comm_df = pd.DataFrame({
        'colonne'        : COLS_COMM,
        'non_null_count' : [stats['non_null'][c] for c in COLS_COMM],
        'non_zero_count' : [stats['commission_non_null'][c] for c in COLS_COMM],
        'somme_totale'   : [stats['commission_somme'][c] for c in COLS_COMM],
    })
    comm_df['pct_non_null'] = (comm_df['non_null_count'] / stats['n_total'] * 100).round(2)
    comm_df['pct_non_zero'] = (comm_df['non_zero_count'] / stats['n_total'] * 100).round(2)
    print(comm_df.to_string(index=False))

    max_non_zero = comm_df['pct_non_zero'].max()
    print('\n--- VERDICT FAISABILITÉ M4 ---')
    if max_non_zero < 1:
        print(f'❌ M4 : NON FAISABLE — commissions non nulles < 1% ({max_non_zero:.2f}%)')
        print('   Les colonnes existent mais sont quasi vides.')
    elif max_non_zero < 5:
        print(f'⚠️  M4 : LIMITÉ — seulement {max_non_zero:.2f}% de commissions non nulles')
        print('   Faisable sur un sous-ensemble de services uniquement.')
    else:
        print(f'✅ M4 : FAISABLE — {max_non_zero:.2f}% de commissions non nulles')
else:
    print('❌ Aucune colonne commission trouvée — M4 NON FAISABLE')

Q2 — COLONNES COMMISSIONS (M4)
                colonne  non_null_count  non_zero_count  somme_totale  pct_non_null  pct_non_zero
       COMMISSIONS_PAID        25456467         9720131  3.193550e+09         100.0         38.18
   COMMISSIONS_RECEIVED        25456467           54838  7.046900e+06         100.0          0.22
    SERVICE_CHARGE_PAID        25456467           30081  2.831031e+07         100.0          0.12
SERVICE_CHARGE_RECEIVED        25456467         5828716  7.765787e+09         100.0         22.90
                  TAXES        25456467               0  0.000000e+00         100.0          0.00

--- VERDICT FAISABILITÉ M4 ---
✅ M4 : FAISABLE — 38.18% de commissions non nulles


## 6. Q3 — Colonnes RECONCILIATION (faisabilité M6)

In [ ]:
print('=' * 60)
print('Q3 — COLONNES RECONCILIATION (M6)')
print('=' * 60)

COLS_RECON = ['RECONCILIATION_BY', 'RECONCILIATION_FOR',
              'EXTERNAL_TRANSACTION_ID', 'TRANSFER_ID', 'TRANSACTION_TAG']
COLS_RECON_PRESENTES = [c for c in COLS_RECON if c in df_peek.columns]

if COLS_RECON_PRESENTES:
    recon_df = pd.DataFrame({
        'colonne'       : COLS_RECON_PRESENTES,
        'non_null_count': [stats['non_null'][c] for c in COLS_RECON_PRESENTES],
    })
    recon_df['pct_non_null'] = (recon_df['non_null_count'] / stats['n_total'] * 100).round(2)
    print(recon_df.to_string(index=False))

    cols_recon_cles = [c for c in ['RECONCILIATION_BY', 'RECONCILIATION_FOR', 'EXTERNAL_TRANSACTION_ID']
                       if c in COLS_RECON_PRESENTES]
    if cols_recon_cles:
        max_recon = max(stats['non_null'][c] / stats['n_total'] * 100 for c in cols_recon_cles)
    else:
        max_recon = 0

    print('\n--- VERDICT FAISABILITÉ M6 ---')
    if not cols_recon_cles:
        print('❌ M6 : NON FAISABLE — colonnes RECONCILIATION absentes')
    elif max_recon < 1:
        print(f'❌ M6 : NON FAISABLE — champs réconciliation < 1% remplis ({max_recon:.2f}%)')
    elif max_recon < 10:
        print(f'⚠️  M6 : LIMITÉ — {max_recon:.2f}% de transactions avec données de réconciliation')
    else:
        print(f'✅ M6 : FAISABLE — {max_recon:.2f}% de transactions réconciliables')
else:
    print('❌ Aucune colonne RECONCILIATION trouvée dans les 80 colonnes')
    print('   → M6 NON FAISABLE avec les données actuelles')
    print('   → Vérifier si ces champs existent sous un autre nom')

Q3 — COLONNES RECONCILIATION (M6)
           colonne  non_null_count  pct_non_null
 RECONCILIATION_BY           89908          0.35
RECONCILIATION_FOR          102514          0.40
       TRANSFER_ID        25456467        100.00
   TRANSACTION_TAG        25456467        100.00

--- VERDICT FAISABILITÉ M6 ---
❌ M6 : NON FAISABLE — champs réconciliation < 1% remplis (0.40%)


## 7. Remplissage MSISDN (faisabilité M2)

In [ ]:
print('=' * 60)
print('MSISDN & USER_ID (M2 — réseaux de mules)')
print('=' * 60)

COLS_ID = ['INITIATOR_MSISDN', 'OTHER_MSISDN', 'SENDER_USER_ID', 'RECEIVER_USER_ID']
COLS_ID = [c for c in COLS_ID if c in df_peek.columns]

id_df = pd.DataFrame({
    'colonne'       : COLS_ID,
    'non_null_count': [stats['non_null'][c] for c in COLS_ID],
})
id_df['pct_non_null'] = (id_df['non_null_count'] / stats['n_total'] * 100).round(2)
print(id_df.to_string(index=False))

print('\n--- VERDICT FAISABILITÉ M2 ---')
# On a besoin d'un couple émetteur/récepteur rempli pour construire le graphe
has_user_ids = all(c in COLS_ID for c in ['SENDER_USER_ID', 'RECEIVER_USER_ID'])
if has_user_ids:
    pct_sender = stats['non_null']['SENDER_USER_ID'] / stats['n_total'] * 100
    pct_recvr  = stats['non_null']['RECEIVER_USER_ID'] / stats['n_total'] * 100
    if pct_sender > 80 and pct_recvr > 80:
        print(f'✅ M2 : FAISABLE sur USER_ID (sender {pct_sender:.0f}%, receiver {pct_recvr:.0f}%)')
        print('   Graphe construit sur SENDER_USER_ID → RECEIVER_USER_ID')
    else:
        print(f'⚠️  M2 : USER_ID partiellement remplis (sender {pct_sender:.0f}%, receiver {pct_recvr:.0f}%)')
else:
    print('⚠️  USER_ID manquants — M2 dépend de MSISDN qui peut être à ~50%')

MSISDN & USER_ID (M2 — réseaux de mules)
         colonne  non_null_count  pct_non_null
INITIATOR_MSISDN           31990          0.13
    OTHER_MSISDN         7187755         28.24
  SENDER_USER_ID        25455705        100.00
RECEIVER_USER_ID        25372367         99.67

--- VERDICT FAISABILITÉ M2 ---
✅ M2 : FAISABLE sur USER_ID (sender 100%, receiver 100%)
   Graphe construit sur SENDER_USER_ID → RECEIVER_USER_ID


## 8. Distribution TRANSFER_STATUS (faisabilité M5)

In [ ]:
print('=' * 60)
print('TRANSFER_STATUS (M5 — prédiction échec)')
print('=' * 60)

statut_df = pd.DataFrame([
    {'statut': k, 'count': v, 'pct': v/stats['n_total']*100}
    for k, v in sorted(stats['statut_counts'].items(), key=lambda x: -x[1])
])
print(statut_df.to_string(index=False))

# Taux d'échec (TF)
n_tf = stats['statut_counts'].get('TF', 0)
pct_tf = n_tf / stats['n_total'] * 100

print(f'\nTransactions en échec (TF) : {n_tf:,} ({pct_tf:.2f}%)')
print('\n--- VERDICT FAISABILITÉ M5 ---')
if pct_tf < 0.5:
    print(f'⚠️  M5 : Déséquilibre fort ({pct_tf:.2f}% TF) → SMOTE ou scale_pos_weight obligatoire')
elif pct_tf < 5:
    print(f'✅ M5 : FAISABLE — {pct_tf:.2f}% TF, class_weight=balanced suffit')
elif pct_tf > 40:
    print(f'⚠️  M5 : Taux TF très élevé ({pct_tf:.2f}%) — vérifier si TF = vrai échec')
else:
    print(f'✅ M5 : FAISABLE — {pct_tf:.2f}% TF, distribution équilibrée')

TRANSFER_STATUS (M5 — prédiction échec)
statut    count       pct
    TS 23967725 94.151812
    TF  1478546  5.808135
   TPI    10017  0.039350
    TI      167  0.000656
    A1       12  0.000047

Transactions en échec (TF) : 1,478,546 (5.81%)

--- VERDICT FAISABILITÉ M5 ---
✅ M5 : FAISABLE — 5.81% TF, distribution équilibrée


## 9. Synthèse globale — remplissage de toutes les colonnes

In [ ]:
remplissage = pd.DataFrame([
    {'colonne': c, 'non_null': stats['non_null'][c],
     'pct_non_null': stats['non_null'][c]/stats['n_total']*100}
    for c in colonnes_a_analyser
]).sort_values('pct_non_null', ascending=False)

print('=== REMPLISSAGE DE TOUTES LES COLONNES ANALYSÉES ===')
print(remplissage.to_string(index=False))

# Sauvegarde
remplissage.to_csv(OUTPUT_DIR / 'diagnostic_remplissage.csv', index=False)
print(f"\n✅ Sauvegardé : {OUTPUT_DIR / 'diagnostic_remplissage.csv'}")

=== REMPLISSAGE DE TOUTES LES COLONNES ANALYSÉES ===
                colonne  non_null  pct_non_null
        TRANSFER_STATUS  25456467    100.000000
SERVICE_CHARGE_RECEIVED  25456467    100.000000
         SENDER_PRE_BAL  25456467    100.000000
        SENDER_POST_BAL  25456467    100.000000
       COMMISSIONS_PAID  25456467    100.000000
           SERVICE_TYPE  25456467    100.000000
                  TAXES  25456467    100.000000
    SERVICE_CHARGE_PAID  25456467    100.000000
     TRANSACTION_AMOUNT  25456467    100.000000
        TRANSACTION_TAG  25456467    100.000000
            TRANSFER_ID  25456467    100.000000
             CREATED_ON  25456467    100.000000
   COMMISSIONS_RECEIVED  25456467    100.000000
         SENDER_USER_ID  25455705     99.997007
      SENDER_ACC_STATUS  25374067     99.676310
       RECEIVER_USER_ID  25372367     99.669632
    RECEIVER_ACC_STATUS  25371605     99.666639
            SENDER_CITY  25128303     98.710882
           OTHER_MSISDN   7187755  

## 10. VERDICT FINAL — faisabilité des 6 modèles

In [ ]:
print('=' * 65)
print('VERDICT FINAL — FAISABILITÉ DES 6 MODÈLES')
print('=' * 65)
print(f"Base analysée : {stats['n_total']:,} transactions sur {nb_jours} jour(s)")
print(f"Période       : {stats['date_min']} → {stats['date_max']}")
print()

verdicts = {}

# M1 — toujours faisable (non supervisé)
verdicts['M1'] = ('✅ FAISABLE', 'Isolation Forest — déjà développé (v2)')

# M2 — dépend des USER_ID
if 'SENDER_USER_ID' in colonnes_a_analyser:
    pct = stats['non_null']['SENDER_USER_ID']/stats['n_total']*100
    verdicts['M2'] = ('✅ FAISABLE' if pct > 80 else '⚠️ LIMITÉ',
                      f'Graphe sur USER_ID — {nb_jours}j de données')
else:
    verdicts['M2'] = ('⚠️ À VÉRIFIER', 'Dépend du remplissage MSISDN/USER_ID')

# M3 — dépend du nb de jours
if nb_jours < 7:
    verdicts['M3'] = ('❌ NON FAISABLE', f'{nb_jours}j — LSTM impossible (besoin 30j+)')
elif nb_jours < 30:
    verdicts['M3'] = ('⚠️ RISQUÉ', f'{nb_jours}j — reformuler en règles + scoring simple')
else:
    verdicts['M3'] = ('✅ FAISABLE', f'{nb_jours}j suffisants pour LSTM')

# M4 — dépend des commissions
if COLS_COMM:
    max_nz = max(stats['commission_non_null'][c]/stats['n_total']*100 for c in COLS_COMM)
    if max_nz < 1:
        verdicts['M4'] = ('❌ NON FAISABLE', f'Commissions {max_nz:.2f}% non nulles')
    elif max_nz < 5:
        verdicts['M4'] = ('⚠️ LIMITÉ', f'Commissions {max_nz:.2f}% — sous-ensemble services')
    else:
        verdicts['M4'] = ('✅ FAISABLE', f'Commissions {max_nz:.2f}% non nulles')
else:
    verdicts['M4'] = ('❌ NON FAISABLE', 'Colonnes commissions absentes')

# M5 — dépend de TF
verdicts['M5'] = ('✅ FAISABLE' if pct_tf >= 0.5 else '⚠️ DÉSÉQUILIBRÉ',
                  f'{pct_tf:.2f}% TF — LightGBM' + (' + SMOTE' if pct_tf < 0.5 else ''))

# M6 — dépend de RECONCILIATION
cols_recon_cles = [c for c in ['RECONCILIATION_BY', 'RECONCILIATION_FOR', 'EXTERNAL_TRANSACTION_ID']
                   if c in colonnes_a_analyser]
if not cols_recon_cles:
    verdicts['M6'] = ('❌ NON FAISABLE', 'Colonnes RECONCILIATION absentes')
else:
    max_r = max(stats['non_null'][c]/stats['n_total']*100 for c in cols_recon_cles)
    if max_r < 1:
        verdicts['M6'] = ('❌ NON FAISABLE', f'Réconciliation {max_r:.2f}% remplie')
    else:
        verdicts['M6'] = ('✅ FAISABLE' if max_r > 10 else '⚠️ LIMITÉ',
                          f'Réconciliation {max_r:.2f}% remplie')

noms = {
    'M1': 'Détection fraude',
    'M2': 'Réseaux mules',
    'M3': 'SIM Swap',
    'M4': 'Anomalies commissions',
    'M5': 'Prédiction échec',
    'M6': 'Réconciliation',
}
for m in ['M1','M2','M3','M4','M5','M6']:
    verdict, detail = verdicts[m]
    print(f'{m} — {noms[m]:<22} {verdict:<18} {detail}')

VERDICT FINAL — FAISABILITÉ DES 6 MODÈLES
Base analysée : 25,456,467 transactions sur 32 jour(s)
Période       : 2025-08-29 10:35:20 → 2025-09-30 23:59:59

M1 — Détection fraude       ✅ FAISABLE         Isolation Forest — déjà développé (v2)
M2 — Réseaux mules          ✅ FAISABLE         Graphe sur USER_ID — 32j de données
M3 — SIM Swap               ✅ FAISABLE         32j suffisants pour LSTM
M4 — Anomalies commissions  ✅ FAISABLE         Commissions 38.18% non nulles
M5 — Prédiction échec       ✅ FAISABLE         5.81% TF — LightGBM
M6 — Réconciliation         ❌ NON FAISABLE     Réconciliation 0.40% remplie


## 11. Conversion Parquet propre (dataset ML)

Une fois le diagnostic validé, on convertit le CSV en Parquet (compression + lecture 10× plus rapide).  
**Lance cette cellule uniquement après avoir validé les verdicts ci-dessus.**

In [ ]:
# Conversion CSV → Parquet par chunks (toutes colonnes)
CONVERTIR_PARQUET = False  # passer à True quand prêt

if CONVERTIR_PARQUET:
    import pyarrow as pa
    import pyarrow.parquet as pq

    parquet_path = OUTPUT_DIR / 'OM_full.parquet'
    writer = None
    debut = time.time()

    reader = pd.read_csv(CSV_PATH, sep=SEPARATEUR, encoding=ENCODING,
                         chunksize=CHUNK_SIZE, low_memory=False)

    for i, chunk in enumerate(reader, 1):
        # Typage minimal : dates + numériques
        if COL_DATE in chunk.columns:
            chunk[COL_DATE] = pd.to_datetime(chunk[COL_DATE], errors='coerce', dayfirst=True)
        table = pa.Table.from_pandas(chunk)
        if writer is None:
            writer = pq.ParquetWriter(parquet_path, table.schema)
        writer.write_table(table)
        if i % 5 == 0:
            print(f'  Chunk {i} converti | {time.time()-debut:.0f}s')

    if writer:
        writer.close()
    taille = parquet_path.stat().st_size / (1024**3)
    print(f'\n✅ Parquet créé : {parquet_path} ({taille:.2f} Go)')
    print(f'   Compression vs CSV : {os.path.getsize(CSV_PATH)/parquet_path.stat().st_size:.1f}×')
else:
    print('Conversion Parquet désactivée. Passe CONVERTIR_PARQUET=True quand prêt.')

Conversion Parquet désactivée. Passe CONVERTIR_PARQUET=True quand prêt.
